# Component 2 - Model Experiments

Compares models for predicting each prop stat (points, rebounds, assists, threes), using `scripts/model_experiments.py` on the feature files (run *Player Stats Ingestion* and *Feature Engineering* first). Takes 5-10 minutes.

**Split (by time, never random):** train on last season before February, tune on the rest of last season, test on this season so far after refitting on both.

**Scores:**
- **MAE**: average miss of the predicted number.
- **Log score**: how likely the actual result was under the predicted distribution (each prediction becomes a negative binomial). Lower is better; it rewards getting the spread right, not just the middle.
- **Brier (over proxy line)**: accuracy of P(over) at a line near the player's 10-game average.

**Against real lines** (historical odds from `fetch_historical_odds.py`, one snapshot ~30 min before each game):
- **tune** bets: last season's games with odds, priced with models trained only on games before February.
- **test** bets: this season's games with odds, priced with the refit models.
- The blend weight between model and Vegas (lowest Brier) and an ev cutoff (best return, 100+ bets) are chosen on **tune** only, then applied unchanged to **test**.

In [ ]:
LEAGUE = "nba"

In [ ]:
import sys
import warnings
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "model_experiments.py").exists())
sys.path.insert(0, str(root / "scripts"))

import importlib
import fetch_player_stats, build_features, model_experiments as me
for m in (fetch_player_stats, build_features, me):
    importlib.reload(m)  # pick up script edits without restarting the kernel

import pandas as pd
warnings.filterwarnings("ignore")
ORDER = ["Last-10 average", "Last-20 average", "Ridge regression", "Poisson regression", "Random forest",
         "Gradient boosting", "Gradient boosting (Poisson)", "Ensemble (RF + GB)"]
TARGETS = ["pts", "reb", "ast", "fg3m"]

## Run

In [ ]:
results, bets, parts, preds, tuning = me.run(LEAGUE)

## Test season results

In [ ]:
def table(metric, split="test"):
    return results[results["split"] == split].pivot(index="model", columns="target", values=metric).loc[ORDER, TARGETS].round(4)

for metric in ("MAE", "Log score", "Brier (over proxy line)"):
    print(metric)
    display(table(metric))

## Validation results (last season, Feb onward)

In [ ]:
for metric in ("MAE", "Log score", "Brier (over proxy line)"):
    print(metric)
    display(table(metric, "valid"))

## Tuning on last season (tune split)

Blend weight: share of the final probability that comes from the model (0 = Vegas only). Chosen by lowest Brier on tune main lines.

In [ ]:
print("Chosen blend weight:", tuning["blend_weight"])
tuning["weights"].round(4)

EV cutoff at the chosen weight: bet only when ev is above it. Chosen by best return with at least 100 tune bets.

In [ ]:
print("Chosen ev cutoff:", tuning["ev_cutoff"])
tuning["cutoffs"].round(4)

## Test season results against real lines

Nothing here was used to choose the settings above.

In [ ]:
evals = me.add_eval_columns(bets, tuned_ev=tuning["ev_cutoff"])
for split in ("tune", "test"):
    print(f"Strategies, {split} main lines")
    display(me.strategy_summary(evals, split=split, tuned_ev=tuning["ev_cutoff"]).round(3))

In [ ]:
print("Test: ev cutoffs and blend weights (reference only)")
display(me.ev_cutoffs(bets, split="test").round(4))
display(me.blend_sensitivity(bets, split="test").round(4))

Does higher ev mean better results? Positive-ev main-line bets grouped by ev:

In [ ]:
for split in ("tune", "test"):
    print(split)
    display(me.ev_buckets(bets, split=split).round(3))

Brier score of the chance of winning, main lines quoted by 3+ books (lower is better):

In [ ]:
for split in ("tune", "test"):
    s = me.vegas_summary(bets, main_only=True, min_books=3, split=split)
    print(f"{split}: {s.groupby('market')['bets'].first().to_dict()}")
    display(s.pivot(index="model", columns="market", values="Brier").round(4))

## Bet by bet

One row per bet: the Over and the Under of each player prop are separate rows. `split` says whether it's a tune or test bet.

- **books / main_line**: books quoting that exact line; the line most books quote for that player and market.
- **best_price / best_book / breakeven_p**: best American odds for this side, where, and the win rate it needs.
- **vegas_p_win**: the books' average no-vig chance this side wins.
- **model_mean / model_p_win**: the ensemble's predicted stat and its chance this side wins.
- **final_p_win**: model blended with Vegas at the tuned weight (`blend_weight`). Used for **ev**.
- **edge**: model minus Vegas chance of winning. **ev**: expected profit per $1 at the best price.
- **actual / outcome**: the stat and whether the bet won, lost or pushed.

In [ ]:
# Test bets the blend likes most
bets[bets["split"] == "test"].sort_values("ev", ascending=False).head(25)

In [ ]:
PLAYER = "Josh Giddey"
bets[bets["player"] == PLAYER]

## Calibration

When the model says 60% over, does it go over about 60% of the time? Test season, proxy lines, all stats pooled. The blend with Vegas is what corrects the overconfidence for betting.

In [ ]:
me.calibration(parts, preds, me.MAIN_MODEL).round(3)

## Chosen settings and dispersion

In [ ]:
results[results["split"] == "test"].pivot(index="model", columns="target", values="settings").loc[ORDER[2:7], TARGETS]

In [ ]:
results.to_csv(me.OUT_DIR / f"{LEAGUE}_model_results.csv", index=False)
if bets is not None:
    bets.to_csv(me.OUT_DIR / f"{LEAGUE}_vegas_bets.csv", index=False)
    me.add_eval_columns(bets, tuned_ev=tuning["ev_cutoff"]).to_csv(me.OUT_DIR / f"{LEAGUE}_vegas_bets_eval.csv", index=False)